In [1]:
!pip install transformers pandas sentencepiece

In [2]:
from google.colab import files
uploaded = files.upload()  # lehce1_dataset.pkl upload

Saving lehce1_dataset.pkl to lehce1_dataset.pkl


In [3]:
import pandas as pd
import pickle
import torch
from transformers import MarianMTModel, MarianTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("lehce1_dataset.pkl", "rb") as f:
    df = pickle.load(f)

model_name = "Helsinki-NLP/opus-mt-pl-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)

def translate_batch(texts, batch_size=8):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model.generate(
            **inputs,
            num_beams=2,
            no_repeat_ngram_size=4
        )
        translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        results.extend(translations)
    return results

pl_texts = df['transcript'].astype(str).tolist()
en_texts = translate_batch(pl_texts)

df = pd.DataFrame({
    'gender': df['gender'].tolist(),
    'text': en_texts
})

df

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


,gender,text
0,female,"Good morning, sir. I have such a kind request ..."
1,female,clinic
2,female,I'm listening.
3,female,"Well, I'll have to come too because I live her..."
4,female,I'm going to go to Górczewska and from there I...
...,...,...
19594,female,save
19595,male,it was also a time of instant careers and prom...
19596,male,small letters
19597,male,God's way groaned Waldemar


In [4]:
with open("lehce1-engdataset.pkl", "wb") as f:
    pickle.dump(df, f)

files.download("lehce1-engdataset.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>